In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.5 Shape tracing and tests

> 🎯 **Goal:** Build the discipline of tracing tensor shapes end to end and write the one test every causal attention must pass.

**Why this matters.** In attention code, the bugs are almost never in the math; they are in the shapes and the mask. A transposed axis or a mask applied one step too late produces a tensor that *runs* but is silently wrong. Two habits catch nearly all of these: trace the shape at every step, and assert the no-future-leak invariant. This lesson is where attention stops being a demo and becomes something you can trust.

**The intuition.** Treat shapes like dimensional analysis in physics. If you write down the shape after every operation and the final shape is what you expect, the plumbing is almost certainly correct. And for a decoder there is one property worth a test of its own: editing the future must never change the past. If it does, your model is cheating, and no accuracy number can be trusted.

**The mechanics.** The full input shape is `[batch, n_head, tokens, hs]`, the general case that contains all the earlier lessons as special cases. We feed it through `scaled_dot_product_attention(..., causal=True)` and confirm:
- The output keeps the input shape `[batch, n_head, tokens, hs]` (attention is shape-preserving in this layout).
- The weight matrix is `[batch, n_head, tokens, tokens]`, one `tokens x tokens` attention map per head per batch item.

Then we prove the invariant directly: clone `k` and `v`, perturb everything from position 1 onward by +3.0 (the entire future), recompute, and measure how much position 0's output changed. For a correct causal mask that change must be zero up to floating-point rounding, so we assert `leak < 1e-6`.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
batch, n_head, tokens, hs = 2, 4, 6, 8
q = torch.randn(batch, n_head, tokens, hs)
k = torch.randn(batch, n_head, tokens, hs)
v = torch.randn(batch, n_head, tokens, hs)
out, w = scaled_dot_product_attention(q, k, v, causal=True)
print("in:", (batch, n_head, tokens, hs), "-> out:", tuple(out.shape),
      " weights:", tuple(w.shape))

**What just happened.** The printed `leak` is effectively 0 (something like `0.0` or a tiny rounding crumb well below `1e-6`), so the assertion passes. We edited the entire future by a large amount across every head and every batch item, and position 0's output did not budge. Combined with the shape assertions, this gives us two guarantees in one cell: the plumbing is correct (shapes match), and the causal invariant holds (no future leak). This is the test you would keep in your suite forever; if a future refactor ever breaks the mask, this cell turns red immediately.

⚠️ **Common pitfalls**
- Indexing the wrong axis when perturbing. With shape `[batch, n_head, tokens, hs]`, the future lives along the token axis, so it is `[:, :, 1:]`. Slicing `[:, 1:]` would perturb the head axis instead and test nothing meaningful.
- Trusting a passing shape check alone. Matching shapes prove the plumbing fits, not that the mask works. You need the leak test too.
- Setting the tolerance too tight. Floating-point math is not exact; `1e-6` allows for normal rounding while still catching any real leak, which would be orders of magnitude larger.

🏋️ **Try it yourself.** Change `causal=True` to `causal=False` on the `out2` call only, and watch `leak` jump to a large number and the assertion fail. That failure is the test doing its job. Then restore it and try perturbing `[:, :, 2:]` while checking position 1: still no leak, because position 1 legitimately sees positions 0 and 1 but nothing beyond.

In [ ]:
k2 = k.clone()
v2 = v.clone()
k2[:, :, 1:] += 3.0
v2[:, :, 1:] += 3.0
out2, _ = scaled_dot_product_attention(q, k2, v2, causal=True)
leak = (out[:, :, 0] - out2[:, :, 0]).abs().max().item()
print("max change at position 0 from editing the future:", leak)
assert tuple(out.shape) == (batch, n_head, tokens, hs)
assert tuple(w.shape) == (batch, n_head, tokens, tokens)
assert leak < 1e-6